# Notebook de Tratamento (13cat)

Parte dos 7 CSVs do notebook 1 e aplica os tratamentos, na versão que **mantém as 13 categorias originais de `losses`**.

**Pré-requisito:** rodar o notebook 1 antes.

> Existem dois notebooks de tratamento idênticos, que diferem **apenas** no tratamento de `losses`: `tratamento_datasets_13cat` e `tratamento_datasets_perfil`. Rode um ou outro (ou ambos, para comparar) — cada um gera seu próprio conjunto de arquivos de saída.

### Tratamentos aplicados
1. Excluir `Oven RTD Connector Body` do Mainback (sensor com defeito).
2. Excluir `AC Blower Inlet Max` e `AC Blower Inlet Avg` do Main Vent Sys (92,9% de nulos).
3. Remover componentes sem nenhuma medição (colunas 100% nulas) e ajustar a tabela de limites.
4. **One-Hot Encoding** das features (valores 0/1), removendo `run_idx`.

Os IGBTs (1–16) e Reso Caps (1–8) **não são agregados** — a média mascararia estouros individuais de limite.

### Tratamento de `losses` nesta versão
A coluna `losses` é usada **como está**, com suas 13 categorias originais (após padronização do notebook 1). Um ensaio sem registro de losses (`NaN`) fica com todas as colunas de losses zeradas — situação que ocorre em 6 ensaios do modo Broil.

Esta é a abordagem que segue a orientação original do professor. A versão alternativa (`tratamento_datasets_perfil`) agrupa as categorias para comparação.


## 1. Configuração e leitura

Carregamos os 7 datasets gerados pelo notebook 1. O filtro no `glob` ignora arquivos com sufixo `_13cat` ou `_perfil` — assim, se este notebook já tiver rodado antes e deixado suas saídas na pasta, ele não tenta reprocessá-las por engano (foi um erro que já aconteceu).

Cada dataset vira uma entrada no dicionário `datasets`, identificada pelo nome da área.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import pandas as pd
import glob

FEATURES = ["blower_direction", "housing", "board_position", "run_case",
            "losses", "deflector_hole", "material_coil_support"]

# Atributos fisicos extraidos do motivo do ensaio, na preparacao. Ja vem em escala
# binaria, de modo que nao passam pelo One-Hot; seguem como estao.
EXTRAS = ["cfg_vazao_declarada", "cfg_vazao_alta", "cfg_cobertura_dissipador",
          "cfg_duto_alterado", "cfg_bandeja_alterada", "cfg_deflector_variante"]

arquivos = sorted([f for f in glob.glob("dataset_*.csv")
                   if "_13cat" not in f and "_perfil" not in f])
print("Datasets encontrados:", len(arquivos))

datasets = {}
for arq in arquivos:
    nome = arq.replace("dataset_", "").replace(".csv", "")
    datasets[nome] = pd.read_csv(arq)
    print(f"  {nome:16s} -> {datasets[nome].shape}")


## 2. Excluir `Oven RTD Connector Body` (Mainback)

Esse sensor tem todas as leituras entre 198 °C e 273 °C, sempre acima do limite de segurança de 150 °C. O padrão é consistente com **defeito de calibração** — o sensor mede errado, não é o componente que está de fato tão quente. Mantê-lo ensinaria o modelo a prever temperaturas fisicamente inválidas. A exclusão afeta só o Mainback.

In [ ]:
if "Oven RTD Connector Body" in datasets["mainback"].columns:
    print(datasets["mainback"]["Oven RTD Connector Body"].describe().round(2))
    datasets["mainback"] = datasets["mainback"].drop(columns=["Oven RTD Connector Body"])
    print("\nExcluído. Novo shape:", datasets["mainback"].shape)


## 3. Excluir colunas com excesso de nulos (Main Vent Sys)

`AC Blower Inlet Max` e `AC Blower Inlet Avg` têm 52 de 56 valores nulos (92,9%). Com apenas 4 leituras cada, não há informação suficiente para o modelo aprender qualquer padrão, e a alta proporção de vazios pode prejudicar a predição. Por isso as duas colunas são removidas do Main Vent Sys.

In [ ]:
cols = [c for c in ["AC Blower Inlet Max", "AC Blower Inlet Avg"]
        if c in datasets["main_vent_sys"].columns]
if cols:
    datasets["main_vent_sys"] = datasets["main_vent_sys"].drop(columns=cols)
    print("Excluídas:", cols, "->", datasets["main_vent_sys"].shape)


## 4. Remover componentes sem nenhuma medição

Alguns componentes não têm leitura de temperatura em nenhum dos 56 ensaios — aparecem como coluna inteiramente nula (o notebook 1 os preserva de propósito, para que essa ausência fique visível aqui em vez de sumir em silêncio).

Um componente sem nenhuma medição não pode ser aprendido nem avaliado, então ele é removido tanto dos datasets quanto da tabela de limites (onde a linha ficaria órfã na comparação pós-predição). A varredura é dinâmica: pega qualquer coluna 100% nula, sem precisar listar nomes.

In [ ]:
limites = pd.read_csv("limites_ativos.csv")
removidos = []
for nome, df in datasets.items():
    alvos = [c for c in df.columns if c not in FEATURES + EXTRAS + ["run_idx"]]
    vazios = [c for c in alvos if df[c].isna().all()]
    if vazios:
        datasets[nome] = df.drop(columns=vazios)
        removidos += vazios
        print(f"  {nome}: removidos {vazios}")
if removidos:
    limites = limites[~limites["tc_name"].isin(removidos)]
else:
    print("Nenhum componente 100% nulo.")
print("Componentes na tabela de limites:", len(limites))


## 5. Manter as 13 categorias de `losses`

Nesta versão, a coluna `losses` segue **como está** (já padronizada pelo notebook 1), conforme a orientação original do professor. Nenhuma categoria é agrupada.

Um ensaio **sem registro** de losses (`NaN`) ficará, no one-hot da próxima seção, com todas as colunas de losses em 0. Isso é diferente de `no losses` (perda zero, caso do Pyro), que tem sua própria coluna — os dois não se confundem.

A célula abaixo só imprime as categorias presentes, para conferência.

In [ ]:
# Conferência: categorias presentes
ex = datasets["coil_temps"]["losses"]
print("Categorias de losses (coil_temps):", ex.nunique(dropna=True), "categorias")
print(ex.value_counts(dropna=False).to_string())


## 6. One-Hot Encoding

Cada coluna de texto (feature de design) é transformada em várias colunas de **0 e 1**, uma por categoria — porque o modelo só faz conta com número, não entende a palavra "pyro" ou "simone".

Duas colunas ficam de fora: `run_idx` (é só um identificador, não uma característica do design) e as colunas de componentes (são o alvo da predição). O `run_idx` é **removido agora**, antes do treino, para não haver risco de entrar como feature por engano.

O `.astype(int)` garante que o resultado seja 0/1 e não `True`/`False` (versões recentes do pandas geram booleano por padrão). Ensaios com `losses` ausente ficam com todas as colunas de losses em 0.

In [ ]:
SUFIXO = "_13cat"

for nome, df in datasets.items():
    alvos = [c for c in df.columns if c not in FEATURES + EXTRAS + ["run_idx"]]
    dummies = pd.get_dummies(df[FEATURES], columns=FEATURES, dummy_na=False).astype(int)
    presentes = [c for c in EXTRAS if c in df.columns]
    out = pd.concat([dummies, df[presentes], df[alvos]], axis=1)   # run_idx sai aqui
    arq = f"dataset_{nome}{SUFIXO}.csv"
    out.to_csv(arq, index=False)
    n_losses = len([c for c in out.columns if c.startswith("losses_")])
    print(f"  {arq:34s} -> {out.shape}  ({n_losses} colunas de losses)")

limites.to_csv("limites_ativos.csv", index=False)


## 7. Conferência final

Duas verificações rápidas de sanidade: que cada ensaio tem no máximo uma categoria de losses marcada (soma 0 quando não há registro, 1 quando há), e que as colunas do one-hot estão como números inteiros (0/1), não booleanos.

In [ ]:
# Um ensaio por vez: soma das colunas de losses deve ser 0 (sem registro) ou 1 (com registro)
df_check = pd.read_csv(f"dataset_coil_temps{SUFIXO}.csv")
losses_cols = [c for c in df_check.columns if c.startswith("losses_")]
soma = df_check[losses_cols].sum(axis=1)
print("Colunas de losses:", losses_cols)
print(f"Ensaios com registro de losses (soma=1): {(soma==1).sum()}")
print(f"Ensaios sem registro (soma=0):          {(soma==0).sum()}")
print()
# Valores 0/1, não booleanos
print("Tipo das colunas one-hot:", df_check[losses_cols].dtypes.unique().tolist())


## 8. Resumo

Saídas: `dataset_<area>_13cat.csv` (7 arquivos) + `limites_ativos.csv` filtrado.

| Tratamento | Onde |
|---|---|
| Oven RTD Connector Body excluído | Mainback |
| AC Blower Inlet Max/Avg excluídos | Main Vent Sys |
| Componentes 100% nulos removidos | Todos + limites |
| IGBTs e Reso Caps mantidos individuais | IPC Comp |
| One-Hot (0/1), run_idx removido | Todos |
| **losses: 13 categorias** | Todos |

> Housing "open - updated" já foi unificado no notebook 1. Os nulos restantes estão nas colunas-alvo (temperaturas) e serão tratados na fase de modelagem com o professor.
